# kuafu-sysid — usage example

End-to-end demo: generate a tiny synthetic dataset, train every model from
`sysid_train.yaml`, evaluate a pinned model from `sysid_select.yaml`, and get
forecasts for downstream use.

**Run this notebook from the `examples/` directory** (the configs use relative
paths). See `../docs/modeling-guide.md` for how to choose lags and the
time-feature encoding.

## 1. Make a demo dataset

Hourly series where the target `y` depends on a driver `load` plus a diurnal
shape. `load_fc` is a known-ahead forecast of `load` (the realistic exog you
feed at inference). Written to `data/demo.parquet` — the path the config names.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

idx = pd.date_range("2024-01-01", periods=24 * 90, freq="h", tz="Europe/Zurich")
rng = np.random.default_rng(0)
hour = idx.hour
load = 50 + 30 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 3, len(idx))
y = 10 + 0.5 * load + 5 * np.sin(2 * np.pi * hour / 24 + 0.5) + rng.normal(0, 2, len(idx))
load_fc = load + rng.normal(0, 1, len(idx))   # known-ahead forecast of load
df = pd.DataFrame({"y": y, "load": load, "load_fc": load_fc}, index=idx)

Path("data").mkdir(exist_ok=True)
df.to_parquet("data/demo.parquet")
df.head()

## 2. Train every model in the config

`train()` reads `sysid_train.yaml`, builds features (selected `lags` + cyclical
time features + `load_fc` as known-ahead exog), fits each model, and saves them
under `models/demo/` with a feature-hash name + `_config.json` sidecar +
`_latest.json` pointer. It returns per-horizon metrics per model.

In [ ]:
from kuafu_sysid import (
    TrainConfig, SelectionConfig, train, evaluate, load_forecaster, plot_error_by_horizon,
)

metrics = train(TrainConfig.from_yaml("sysid_train.yaml"))
plot_error_by_horizon(metrics, metric="rmse")
{m: round(d["rmse"].mean(), 3) for m, d in metrics.items()}   # mean RMSE per method

## 3. Evaluate a pinned model and get forecasts

`sysid_select.yaml` pins the `demo` role to a saved model (here `feature_hash`
is blank, so the loader uses `_latest.json`). `evaluate()` reports per-horizon
error; `load_forecaster().predict()` returns the wide forecast DataFrame the
optimization would consume.

In [ ]:
sel = SelectionConfig.from_yaml("sysid_select.yaml")
res = evaluate(sel, "demo", df)
print(res.metrics.round(3).head())

forecast = load_forecaster(sel, "demo").predict(df)   # index=origin, cols=y_h_1..H
forecast.head()

## Try it yourself

- Prune `lags` in `sysid_train.yaml` (e.g. `[0, 1, 24]`) — fewer columns, faster,
  often as accurate. Each change is a new `feature_hash`, so models coexist.
- Switch `time_features.encoding` to `onehot`, or toggle individual components.
- Pin a specific model by copying its hash from `models/demo/_latest.json` into
  `sysid_select.yaml`.

See `../docs/modeling-guide.md` for the trade-offs.